In [27]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [28]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0, google_api_key=os.getenv("GEMINI_API_KEY"))

In [4]:
from typing import Annotated
import operator
from langgraph.graph import MessagesState

class State(MessagesState):
    summary: str

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

def call_model(state:State):
    summary=state.get("summary","")
    
    if summary:
        system_message = f"summary of conversation earlier: {summary}"
        
        messages = [SystemMessage(content=system_message)] + state["messages"]
    else:
        messages = state["messages"]
        
    response = model.invoke(messages)
    return {"messages":response}


In [ ]:
def summarize_conversation(state:State):
    
    summary=state.get("summary","")
    
    if summary:
        summary_message=(
            f"This is summary of the conversation to date: {summary}\n\n"
            "Extend the summary by taking into account the new messages above:\n"
        )
    else:
        summary_message="Create a summary of the conversation above:"
        
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    
    response = model.invoke(messages)
    
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary":response.content, "messages":delete_messages}

In [17]:
from langgraph.graph import END

def should_continue(state:State):
    """Return the next node to execute"""
    
    messages = state["messages"]
    
    if len(messages) >=6:
        return "summarize_conversation"
    else:
        return END

In [25]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node("summarize_conversation", summarize_conversation)

workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_conversation", END)

memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)

In [ ]:
config = {"configurable":{"thread_id":"1"}}

input_message = HumanMessage(content="Hello! How are you?")
output=graph.invoke({"messages":[input_message]}, config=config)

In [ ]:
for m in output["messages"][-1:]:
    m.pretty_print()